In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

def main():
    # =========================================================================
    # NEW: HARDCODED MODEL EXCLUSION LIST
    # =========================================================================
    # Add the exact folder names of any models you want to skip.
    EXCLUDED_MODELS = [
        "google_gemma-3-12b-it"
    ]
    
    # =========================================================================
    # NEW: FIGURE SIZING CONFIGURATION (Adjust for Paper Compactness)
    # =========================================================================
    # Decrease SUBPLOT_WIDTH to make the graphs narrower horizontally.
    # If the image shrinks too much on your paper, lower this number (e.g., 3.0 or 3.5).
    SUBPLOT_WIDTH = 4.0  # Originally 6.0
    SUBPLOT_HEIGHT = 4.5 # Originally 4.5

    # 1. Setup our Base Directories
    in_base_dir = os.path.join("result", "attention_timeseries")
    out_base_dir = os.path.join("result", "process", "attention_timeseries")
    os.makedirs(out_base_dir, exist_ok=True)
    
    print(f"[INFO] Scanning {in_base_dir} for all CSV files...")
    
    search_pattern = os.path.join(in_base_dir, "**", "*.csv")
    all_csvs = glob.glob(search_pattern, recursive=True)
    
    if not all_csvs:
        print(f"[ERROR] No CSVs found in {in_base_dir}")
        return
        
    print(f"[INFO] Found {len(all_csvs)} CSV files. Extracting and binning data...")
    
    # 2. Grouping Dictionary for AGGREGATION
    # Key: (model_name, subject, layer) 
    # Value: list of binned dataframes
    master_data_groups = {}
    
    num_bins = 30
    bins = np.linspace(0, 1, num_bins + 1)
    
    for csv_path in tqdm(all_csvs, desc="Reading & Binning CSVs"):
        rel_path = os.path.relpath(csv_path, in_base_dir)
        parts = rel_path.split(os.sep)
        
        if len(parts) < 4:
            continue
            
        model_name = parts[0]
        
        # Skip this iteration if the model is in our exclusion list
        if model_name in EXCLUDED_MODELS:
            continue
            
        subject = parts[1]
        filename = parts[-1]
        
        if "_layer" not in filename:
            continue
            
        prefix, layer_part = filename.split("_layer")
        layer = layer_part.replace(".csv", "")
        
        last_underscore_idx = prefix.rfind('_')
        if last_underscore_idx != -1:
            prob_id = prefix[:last_underscore_idx]
            variant = prefix[last_underscore_idx + 1:]
            
            df = pd.read_csv(csv_path)
            if len(df) <= 1:
                continue
                
            # Normalize progress
            min_tok = df['token_number_generated'].min()
            max_tok = df['token_number_generated'].max()
            if max_tok == min_tok:
                continue 
                
            df['normalized_progress'] = (df['token_number_generated'] - min_tok) / (max_tok - min_tok)
            
            # Bin the data
            df['bin'] = pd.cut(df['normalized_progress'], bins=bins, include_lowest=True)
            binned_df = df.groupby('bin', observed=False).mean(numeric_only=True).reset_index()
            binned_df['plot_x'] = binned_df['bin'].apply(lambda x: x.mid)
            binned_df = binned_df.dropna(subset=['IAR_normalized'])
            
            if binned_df.empty:
                continue
                
            # Tag the data so we can group it later
            binned_df['variant'] = variant
            binned_df['prob_id'] = prob_id
            
            key = (model_name, subject, layer)
            if key not in master_data_groups:
                master_data_groups[key] = []
            
            master_data_groups[key].append(binned_df[['variant', 'prob_id', 'plot_x', 'IAR_normalized']])
    
    print(f"\n[INFO] Data extracted. Found {len(master_data_groups)} unique (Model, Subject, Layer) combinations to aggregate.")
    
    # =========================================================================
    # 3. MASTER GRID PLOTTING
    # =========================================================================
    # Find all unique layers (just in case you processed multiple layers, we make one grid per layer)
    unique_layers = sorted(list(set([k[2] for k in master_data_groups.keys()])))
    
    for layer in unique_layers:
        print(f"\n[INFO] Generating Master Grid for Layer {layer}...")
        
        # Filter keys for this specific layer
        layer_keys = [k for k in master_data_groups.keys() if k[2] == layer]
        
        if not layer_keys:
            continue
            
        # Determine the grid dimensions
        unique_models = sorted(list(set([k[0] for k in layer_keys])))
        unique_subjects = sorted(list(set([k[1] for k in layer_keys])))
        
        nrows = len(unique_models)
        ncols = len(unique_subjects)
        
        # Dynamically size the figure based on the configured subplot width/height
        fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(SUBPLOT_WIDTH * ncols, SUBPLOT_HEIGHT * nrows), squeeze=False)
        fig.suptitle(f"Image Attention Ratio (Layer {layer})", fontsize=24, fontweight='bold', y=1.02)
        
        for i, model in enumerate(unique_models):
            for j, subject in enumerate(unique_subjects):
                ax = axes[i, j]
                key = (model, subject, layer)
                
                if key not in master_data_groups:
                    # If a specific model/subject combo is missing, hide the axes and skip
                    ax.axis('off')
                    continue
                    
                # Combine and calculate statistics
                dfs = master_data_groups[key]
                master_df = pd.concat(dfs, ignore_index=True)
                num_unique_problems = len(master_df['prob_id'].unique())
                
                agg_df = master_df.groupby(['variant', 'plot_x'])['IAR_normalized'].agg(['mean', 'sem']).reset_index()
                agg_df['ci_lower'] = (agg_df['mean'] - (1.96 * agg_df['sem'])).clip(lower=0.0)
                agg_df['ci_upper'] = (agg_df['mean'] + (1.96 * agg_df['sem'])).clip(upper=1.0)
                
                global_min_y = agg_df['ci_lower'].min()
                global_max_y = agg_df['ci_upper'].max()
                
                variants = agg_df['variant'].unique()
                for variant in sorted(variants):
                    v_data = agg_df[agg_df['variant'] == variant]
                    line = ax.plot(
                        v_data['plot_x'], v_data['mean'], label=variant.capitalize(), 
                        linewidth=2.5, marker='o', markersize=4
                    )
                    ax.fill_between(
                        v_data['plot_x'], v_data['ci_lower'], v_data['ci_upper'], 
                        alpha=0.15, color=line[0].get_color()
                    )

                # --- Subplot Formatting ---
                # Set Column Headers (Top Row Only)
                if i == 0:
                    ax.set_title(f"{subject.capitalize()}\n(n={num_unique_problems} problems)", fontsize=16, fontweight='bold', pad=10)
                else:
                    ax.set_title(f"(n={num_unique_problems} problems)", fontsize=11)
                
                # Set Row Headers (Far Left Column Only)
                if j == 0:
                    ax.set_ylabel(f"{model}\nMean-Normalized IAR", fontsize=14, fontweight='bold')
                
                # Set X-Axis labels (Bottom Row Only)
                # if i == nrows - 1:
                #     ax.set_xlabel("Generation Progress (0 = Start, 1 = End)", fontsize=12)
                
                # Dynamic Y-Axis Zooming
                y_range = global_max_y - global_min_y
                y_padding = y_range * 0.15 if y_range > 0 else 0.1 
                final_min_y = max(0.0, global_min_y - y_padding)
                final_max_y = min(1.0, global_max_y + y_padding)
                ax.set_ylim(final_min_y, final_max_y)
                
                ax.grid(True, linestyle='--', alpha=0.6)
                
                # Add Legend ONLY to the top-right subplot so it doesn't clutter every graph
                if i == 0 and j == ncols - 1:
                    # Tweak legend formatting slightly for tighter spacing
                    ax.legend(title="Language", bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., fontsize=16)

        plt.tight_layout()
        
        # Save
        out_filename = f"MASTER_GRID_layer{layer}.png"
        out_path = os.path.join(out_base_dir, out_filename)
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close()

    print("\n[SUCCESS] Master aggregate grid generated successfully!")

if __name__ == "__main__":
    main()